# 🎭 Mask R-CNN Team Fine-tuning (Cascade Fixed)

**Google Colab Ready - Haar Cascade 로드 문제 수정 버전**

### 사용 방법
1. 런타임 → 런타임 유형 변경 → **GPU (T4 권장)**
2. 아래 셀을 **위에서부터 순서대로** 실행
3. Cell 3 실행 후 나오는 Cloudflare URL을 브라우저에서 열기
4. **팀원 커스텀 학습 & 추론** 모드 사용

### 이번 수정 (v3.1)
- Haar Cascade를 **앱 내부에서 강제 다운로드 + 파일 크기 검증**
- 여러 경로 탐색 강화
- Cascade 로드 성공/실패 시 명확한 안내 메시지


## Cell 1: 패키지 설치 + Haar Cascade 다운로드

In [ ]:
!pip install -q streamlit torch torchvision Pillow matplotlib pycocotools opencv-python-headless albumentations

print("📥 Cloudflare 터널링 도구 다운로드 중...")
!curl -L --silent -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version
print("✅ 설치 완료")

import urllib.request
import os

CASCADE_URL = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
CASCADE_PATHS = [
    "/content/haarcascade_frontalface_default.xml",
    "/tmp/haarcascade_frontalface_default.xml",
    "haarcascade_frontalface_default.xml",
]

for path in CASCADE_PATHS:
    try:
        if not os.path.exists(path) or os.path.getsize(path) < 10000:
            print(f"📥 Haar Cascade 다운로드 중 → {path}")
            urllib.request.urlretrieve(CASCADE_URL, path)
            print(f"✅ 다운로드 완료: {path} ({os.path.getsize(path)} bytes)")
        else:
            print(f"✅ 이미 존재: {path} ({os.path.getsize(path)} bytes)")
    except Exception as e:
        print(f"⚠️ {path} 다운로드 실패: {e}")

print("\n=== Cascade 파일 확인 ===")
for path in CASCADE_PATHS:
    if os.path.exists(path):
        print(f"  ✓ {path} ({os.path.getsize(path)} bytes)")
    else:
        print(f"  ✗ {path}")


## Cell 2: app.py 생성 (Cascade 로드 강화)

In [ ]:
app_code = r'''
import streamlit as st
import torch
import numpy as np
from PIL import Image, ImageDraw
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import io
import os
import cv2
import glob
import urllib.request
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

st.set_page_config(page_title="Mask R-CNN Team Demo (Optimized)", page_icon="🎭", layout="wide")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
st.sidebar.markdown(f"**Device**: `{device}`")

COCO_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

# -------------------- Haar Cascade 로더 (강화 버전) --------------------
def ensure_cascade_file():
    """Cascade 파일이 없으면 강제로 다운로드"""
    CASCADE_URL = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
    candidates = [
        "/content/haarcascade_frontalface_default.xml",
        "/tmp/haarcascade_frontalface_default.xml",
        os.path.join(os.getcwd(), "haarcascade_frontalface_default.xml"),
        "haarcascade_frontalface_default.xml",
    ]

    for path in candidates:
        if os.path.exists(path) and os.path.getsize(path) > 10000:
            return path

    for path in candidates:
        try:
            dirname = os.path.dirname(path)
            if dirname:
                os.makedirs(dirname, exist_ok=True)
            urllib.request.urlretrieve(CASCADE_URL, path)
            if os.path.exists(path) and os.path.getsize(path) > 10000:
                return path
        except Exception:
            continue
    return None


def load_face_cascade():
    """여러 경로를 시도하고, 없으면 다운로드까지 수행"""
    if not hasattr(cv2, 'CascadeClassifier'):
        return None

    cascade_path = ensure_cascade_file()

    candidates = []
    if cascade_path:
        candidates.append(cascade_path)

    candidates.extend([
        "/content/haarcascade_frontalface_default.xml",
        "/tmp/haarcascade_frontalface_default.xml",
        "haarcascade_frontalface_default.xml",
    ])

    try:
        if hasattr(cv2, 'data') and hasattr(cv2.data, 'haarcascades'):
            candidates.append(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    except Exception:
        pass

    search_patterns = [
        '/usr/local/lib/python*/dist-packages/cv2/data/haarcascade_frontalface_default.xml',
        '/usr/lib/python*/dist-packages/cv2/data/haarcascade_frontalface_default.xml',
        '/usr/share/opencv*/haarcascades/haarcascade_frontalface_default.xml',
        '/usr/local/share/opencv*/haarcascades/haarcascade_frontalface_default.xml',
    ]
    for pattern in search_patterns:
        candidates.extend(glob.glob(pattern))

    seen = set()
    unique_candidates = []
    for p in candidates:
        if p and p not in seen:
            seen.add(p)
            unique_candidates.append(p)

    for path in unique_candidates:
        if path and os.path.exists(path) and os.path.getsize(path) > 10000:
            try:
                cascade = cv2.CascadeClassifier(path)
                if cascade is not None and not cascade.empty():
                    return cascade
            except Exception:
                continue
    return None


# -------------------- Dataset --------------------
class TeamFaceDataset(Dataset):
    def __init__(self, root_dir, class_names, transforms=None, max_size=640):
        self.root_dir = Path(root_dir)
        self.class_names = class_names
        self.transforms = transforms
        self.max_size = max_size
        self.samples = []
        self.face_cascade = load_face_cascade()

        print("🔄 얼굴 검출 및 마스크 사전 계산 중... (한 번만 수행)")
        if self.face_cascade is None:
            print("⚠️ Cascade 로드 실패 → 중앙 영역 fallback 사용")
        else:
            print("✅ Cascade 로드 성공")

        for idx, name in enumerate(class_names):
            person_dir = self.root_dir / name
            if not person_dir.exists():
                continue
            for img_path in person_dir.glob("*"):
                if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                    continue

                image = cv2.imread(str(img_path))
                if image is None:
                    continue
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                h, w = image.shape[:2]

                faces = []
                if self.face_cascade is not None:
                    try:
                        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
                        faces = self.face_cascade.detectMultiScale(
                            gray, scaleFactor=1.1, minNeighbors=4, minSize=(40, 40)
                        )
                    except Exception:
                        faces = []

                if faces is None or len(faces) == 0:
                    cx, cy = w // 2, h // 2
                    bw, bh = int(w * 0.4), int(h * 0.5)
                    x1 = max(0, cx - bw // 2)
                    y1 = max(0, cy - bh // 2)
                    x2 = min(w, cx + bw // 2)
                    y2 = min(h, cy + bh // 2)
                    faces = [(x1, y1, x2 - x1, y2 - y1)]

                boxes = []
                masks = []
                for (x, y, fw, fh) in faces:
                    x1, y1, x2, y2 = int(x), int(y), int(x + fw), int(y + fh)
                    boxes.append([x1, y1, x2, y2])

                    mask = np.zeros((h, w), dtype=np.uint8)
                    center = (int((x1 + x2) / 2), int((y1 + y2) / 2))
                    axes = (max(1, int(fw * 0.55)), max(1, int(fh * 0.65)))
                    cv2.ellipse(mask, center, axes, 0, 0, 360, 1, -1)
                    masks.append(mask)

                if len(boxes) > 0:
                    self.samples.append((
                        str(img_path),
                        idx + 1,
                        np.array(boxes, dtype=np.float32),
                        np.stack(masks).astype(np.uint8)
                    ))

        print(f"✅ 사전 계산 완료: {len(self.samples)}장")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, boxes, masks = self.samples[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
        labels_t = torch.as_tensor([label] * len(boxes), dtype=torch.int64)
        masks_t = torch.as_tensor(masks, dtype=torch.uint8)
        area = (boxes_t[:, 3] - boxes_t[:, 1]) * (boxes_t[:, 2] - boxes_t[:, 0])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "masks": masks_t,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": iscrowd
        }

        if self.transforms:
            try:
                transformed = self.transforms(
                    image=image,
                    masks=masks,
                    bboxes=boxes,
                    class_labels=[label] * len(boxes)
                )
                image = transformed["image"]
                target["masks"] = torch.as_tensor(transformed["masks"], dtype=torch.uint8)
                if len(transformed["bboxes"]) > 0:
                    target["boxes"] = torch.as_tensor(transformed["bboxes"], dtype=torch.float32)
                    target["labels"] = torch.as_tensor(transformed["class_labels"], dtype=torch.int64)
                else:
                    target["boxes"] = torch.zeros((0, 4), dtype=torch.float32)
                    target["labels"] = torch.zeros((0,), dtype=torch.int64)
                    target["masks"] = torch.zeros((0, h, w), dtype=torch.uint8)
            except Exception:
                image = T.ToTensor()(image)
        else:
            image = T.ToTensor()(image)

        return image, target


def get_transform(train=True, max_size=640):
    if train:
        return A.Compose([
            A.LongestMaxSize(max_size=max_size, p=1.0),
            A.PadIfNeeded(min_height=max_size, min_width=max_size, border_mode=0),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.HueSaturationValue(p=0.2),
            A.GaussNoise(p=0.15),
            A.Rotate(limit=12, p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], min_visibility=0.3))
    else:
        return A.Compose([
            A.LongestMaxSize(max_size=max_size, p=1.0),
            A.PadIfNeeded(min_height=max_size, min_width=max_size, border_mode=0),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])


def get_model(num_classes):
    weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model = maskrcnn_resnet50_fpn_v2(weights=weights)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model


def collate_fn(batch):
    return tuple(zip(*batch))


# -------------------- UI --------------------
st.title("🎭 Mask R-CNN Instance Segmentation (Speed Optimized)")
mode = st.radio("모드 선택", ["COCO Pretrained Demo", "팀원 커스텀 학습 & 추론"], horizontal=True)

if mode == "COCO Pretrained Demo":
    st.markdown("""
    **Mask R-CNN** (He et al., 2017) – Full COCO pretrained weights
    - **RoIAlign**: 양자화 없이 정밀한 공간 정렬
    - **Mask-Class 분리**: 클래스 독립 Binary Mask (Sigmoid + BCE)
    - **단순 & 범용**: Faster R-CNN 위에 Mask branch 추가
    """)

    @st.cache_resource
    def load_coco_model():
        weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
        model = maskrcnn_resnet50_fpn_v2(weights=weights)
        model.to(device)
        model.eval()
        return model, weights

    with st.spinner("COCO 모델 로딩 중..."):
        model, weights = load_coco_model()
    st.sidebar.success("Full COCO 가중치 로드 성공! (Mask AP ≈ 41.8)")

    st.sidebar.header("⚙️ 추론 파라미터")
    confidence_threshold = st.sidebar.slider("신뢰도 임계값", 0.1, 0.9, 0.5, 0.05)
    show_masks = st.sidebar.checkbox("마스크 표시", value=True)
    show_boxes = st.sidebar.checkbox("바운딩 박스 표시", value=True)
    mask_alpha = st.sidebar.slider("마스크 투명도", 0.1, 0.8, 0.45, 0.05)
    max_detections = st.sidebar.slider("최대 검출 개수", 5, 50, 20)

    if 'active_image' not in st.session_state:
        st.session_state['active_image'] = None

    st.header("1. 이미지 준비")
    input_option = st.radio("이미지 가져오기", ["이미지 업로드", "가상 이미지 생성"], horizontal=True)

    if input_option == "이미지 업로드":
        uploaded_file = st.file_uploader("JPG/PNG 업로드", type=["jpg", "jpeg", "png"])
        if uploaded_file is not None:
            st.session_state['active_image'] = Image.open(uploaded_file).convert("RGB")
    else:
        if st.button("합성 이미지 생성"):
            img_size = 400
            img = Image.new("RGB", (img_size, img_size), color=(30, 30, 40))
            draw = ImageDraw.Draw(img)
            colors = [(220, 80, 80), (80, 180, 100), (80, 120, 220)]
            for i, color in enumerate(colors):
                x1 = 50 + i * 90
                y1 = 80 + i * 40
                draw.rectangle([x1, y1, x1+120, y1+100], fill=color)
                draw.text((x1+10, y1+10), f"Obj {i+1}", fill=(255,255,255))
            st.session_state['active_image'] = img

    if st.session_state['active_image'] is not None:
        st.image(st.session_state['active_image'], caption="분석 대상 이미지", use_container_width=True)
        st.header("2. 추론")
        if st.button("🚀 인스턴스 분할 시작", type="primary"):
            start = time.time()
            current_img = st.session_state['active_image']
            transform = weights.transforms()
            img_tensor = transform(current_img).to(device)

            with torch.no_grad():
                pred = model([img_tensor])[0]

            elapsed = time.time() - start
            st.success(f"추론 완료! {elapsed:.2f}초")

            keep = pred["scores"] > confidence_threshold
            boxes  = pred["boxes"][keep].cpu().numpy()
            scores = pred["scores"][keep].cpu().numpy()
            labels = pred["labels"][keep].cpu().numpy()
            masks  = pred["masks"][keep].cpu().numpy()

            if len(boxes) > max_detections:
                boxes = boxes[:max_detections]
                scores = scores[:max_detections]
                labels = labels[:max_detections]
                masks = masks[:max_detections]

            st.write(f"**탐지된 오브젝트**: {len(boxes)}개")

            st.header("3. 시각화")
            fig, ax = plt.subplots(1, 1, figsize=(12, 12))
            ax.imshow(np.array(current_img))
            np.random.seed(42)
            colors = plt.cm.tab20(np.linspace(0, 1, 20))

            for i, (box, score, label, mask) in enumerate(zip(boxes, scores, labels, masks)):
                x1, y1, x2, y2 = box
                color = colors[label % 20]
                class_name = COCO_NAMES[label] if label < len(COCO_NAMES) else f"cls_{label}"

                if show_boxes:
                    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2.5, edgecolor=color, facecolor="none")
                    ax.add_patch(rect)
                    ax.text(x1, y1-8, f"{class_name} {score:.2f}", color="white", fontsize=10, weight="bold",
                            bbox=dict(facecolor=color, alpha=0.75, pad=2, edgecolor="none"))

                if show_masks:
                    m = mask[0]
                    colored_mask = np.zeros((*m.shape, 4))
                    colored_mask[m > 0.5] = (*color[:3], mask_alpha)
                    ax.imshow(colored_mask)

            ax.axis("off")
            ax.set_title(f"Mask R-CNN (COCO) | {len(boxes)} Instances", fontsize=14)
            plt.tight_layout()
            st.pyplot(fig)

            img_buf = io.BytesIO()
            fig.savefig(img_buf, format="png", bbox_inches="tight", dpi=150)
            img_buf.seek(0)
            st.download_button("📥 결과 이미지 다운로드", data=img_buf, file_name="mask_rcnn_result.png", mime="image/png")

            with st.expander("상세 검출 리스트"):
                for i, (sc, lb) in enumerate(zip(scores, labels)):
                    name = COCO_NAMES[lb] if lb < len(COCO_NAMES) else f"cls_{lb}"
                    st.write(f"{i+1}. **{name}** — Score: {sc:.4f}")

else:
    st.markdown("""
    ### 팀원 4명 얼굴 커스텀 학습 & 추론 (속도 최적화 버전)
    - 각 인원당 **최소 5장**의 얼굴 사진을 업로드하세요.
    - 이니셜은 영문 3자리 (예: LDW, KSH, KKH, LSB)
    - 얼굴 마스크는 **사전 계산**됩니다
    - Mixed Precision + 해상도 제한 + 강한 동결로 **학습 속도 대폭 향상**
    """)

    initials_input = st.text_input("4명의 영문 이니셜 (쉼표로 구분)", value="LDW,KSH,KKH,LSB")
    class_names = [x.strip().upper() for x in initials_input.split(",") if x.strip()]

    if len(class_names) != 4:
        st.warning("정확히 4개의 이니셜을 입력해주세요.")
        st.stop()

    st.success(f"클래스: {class_names}")

    data_root = Path("team_data")
    data_root.mkdir(exist_ok=True)

    st.header("1. 팀원별 얼굴 이미지 업로드 (각 5장 이상)")
    for name in class_names:
        person_dir = data_root / name
        person_dir.mkdir(exist_ok=True)
        files = st.file_uploader(f"**{name}** 얼굴 사진", type=["jpg", "jpeg", "png"], accept_multiple_files=True, key=f"up_{name}")
        if files:
            for f in files:
                img = Image.open(f).convert("RGB")
                save_path = person_dir / f.name
                img.save(save_path)
            st.write(f"✅ {name}: {len(list(person_dir.glob('*')))}장 저장됨")

    st.header("2. 모델 Fine-tuning")
    col1, col2, col3 = st.columns(3)
    with col1:
        epochs = st.number_input("Epochs", min_value=5, max_value=30, value=12)
    with col2:
        batch_size = st.number_input("Batch Size", min_value=1, max_value=4, value=2)
    with col3:
        max_size = st.number_input("Max Image Size", min_value=384, max_value=800, value=640, step=64)

    if st.button("🚀 데이터셋 준비 & Fine-tuning 시작", type="primary"):
        total_imgs = 0
        for name in class_names:
            cnt = len(list((data_root / name).glob("*")))
            total_imgs += cnt
            if cnt < 3:
                st.error(f"{name} 이미지가 너무 적습니다 (최소 3장 권장, 5장 이상 추천)")
                st.stop()

        st.info(f"총 {total_imgs}장으로 학습을 시작합니다... (사전 계산 + AMP 적용)")

        cascade = load_face_cascade()
        if cascade is None:
            st.warning("⚠️ Haar Cascade를 찾지 못했습니다. 중앙 영역 기반 마스크로 진행합니다.")
            st.info("💡 해결 방법: Cell 1을 다시 실행한 뒤, 이 페이지를 새로고침(F5) 해보세요.")
        else:
            st.success("✅ Haar Cascade 로드 성공 → 정확한 얼굴 영역으로 학습합니다.")

        dataset = TeamFaceDataset(
            data_root, class_names,
            transforms=get_transform(train=True, max_size=int(max_size)),
            max_size=int(max_size)
        )
        if len(dataset) == 0:
            st.error("유효한 이미지가 없습니다.")
            st.stop()

        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=2,
            pin_memory=True if device.type == "cuda" else False
        )

        num_classes = len(class_names) + 1
        model = get_model(num_classes)
        model.to(device)

        for name, param in model.named_parameters():
            if "backbone" in name or "fpn" in name:
                param.requires_grad = False

        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=1.5e-4, weight_decay=1e-4)
        lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
        scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

        progress_bar = st.progress(0)
        status_text = st.empty()
        loss_chart = st.empty()
        losses_history = []

        model.train()
        for epoch in range(epochs):
            epoch_loss = 0.0
            for i, (images, targets) in enumerate(loader):
                images = list(img.to(device, non_blocking=True) for img in images)
                targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

                optimizer.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                    loss_dict = model(images, targets)
                    losses = sum(loss for loss in loss_dict.values())

                scaler.scale(losses).backward()
                scaler.step(optimizer)
                scaler.update()

                epoch_loss += losses.item()

            lr_scheduler.step()
            avg_loss = epoch_loss / max(len(loader), 1)
            losses_history.append(avg_loss)

            progress_bar.progress((epoch + 1) / epochs)
            status_text.text(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")
            loss_chart.line_chart(losses_history)

        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "num_classes": num_classes
        }, "custom_team_maskrcnn.pth")

        st.success("✅ Fine-tuning 완료! 모델이 `custom_team_maskrcnn.pth`로 저장되었습니다.")
        st.balloons()

    st.header("3. 단체 사진 추론 (커스텀 모델)")
    if os.path.exists("custom_team_maskrcnn.pth"):
        ckpt = torch.load("custom_team_maskrcnn.pth", map_location=device, weights_only=False)
        custom_class_names = ckpt["class_names"]
        num_classes = ckpt["num_classes"]

        @st.cache_resource
        def load_custom_model():
            model = get_model(num_classes)
            model.load_state_dict(ckpt["model_state_dict"])
            model.to(device)
            model.eval()
            return model

        custom_model = load_custom_model()
        st.sidebar.success(f"커스텀 모델 로드 완료: {custom_class_names}")

        conf_thresh = st.sidebar.slider("커스텀 신뢰도 임계값", 0.1, 0.9, 0.4, 0.05, key="custom_conf")
        show_masks_c = st.sidebar.checkbox("마스크 표시", value=True, key="cmask")
        show_boxes_c = st.sidebar.checkbox("박스 표시", value=True, key="cbox")
        mask_alpha_c = st.sidebar.slider("마스크 투명도", 0.1, 0.8, 0.5, 0.05, key="calpha")

        group_file = st.file_uploader("단체 그룹 사진 업로드", type=["jpg", "jpeg", "png"], key="group")
        if group_file is not None:
            group_img = Image.open(group_file).convert("RGB")
            st.image(group_img, caption="그룹 사진", use_container_width=True)

            if st.button("🔍 팀원 인식 & 세그멘테이션 실행", type="primary"):
                transform = A.Compose([
                    A.LongestMaxSize(max_size=640, p=1.0),
                    A.PadIfNeeded(min_height=640, min_width=640, border_mode=0),
                    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                    ToTensorV2()
                ])
                img_np = np.array(group_img)
                transformed = transform(image=img_np)
                img_tensor = transformed["image"].to(device)

                with torch.no_grad():
                    pred = custom_model([img_tensor])[0]

                keep = pred["scores"] > conf_thresh
                boxes  = pred["boxes"][keep].cpu().numpy()
                scores = pred["scores"][keep].cpu().numpy()
                labels = pred["labels"][keep].cpu().numpy()
                masks  = pred["masks"][keep].cpu().numpy()

                st.write(f"**탐지된 팀원**: {len(boxes)}명")

                fig, ax = plt.subplots(1, 1, figsize=(14, 14))
                ax.imshow(img_np)
                colors = plt.cm.Set1(np.linspace(0, 1, 9))

                for box, score, label, mask in zip(boxes, scores, labels, masks):
                    x1, y1, x2, y2 = box
                    color = colors[(label - 1) % 9]
                    name = custom_class_names[label - 1] if 1 <= label <= len(custom_class_names) else f"cls_{label}"

                    if show_boxes_c:
                        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=3, edgecolor=color, facecolor="none")
                        ax.add_patch(rect)
                        ax.text(x1, y1 - 10, f"{name} {score:.2f}", color="white", fontsize=12, weight="bold",
                                bbox=dict(facecolor=color, alpha=0.85, pad=3, edgecolor="none"))

                    if show_masks_c:
                        m = mask[0]
                        colored_mask = np.zeros((*m.shape, 4))
                        colored_mask[m > 0.5] = (*color[:3], mask_alpha_c)
                        ax.imshow(colored_mask)

                ax.axis("off")
                ax.set_title(f"Custom Team Mask R-CNN | Detected: {len(boxes)}", fontsize=16)
                plt.tight_layout()
                st.pyplot(fig)

                img_buf = io.BytesIO()
                fig.savefig(img_buf, format="png", bbox_inches="tight", dpi=150)
                img_buf.seek(0)
                st.download_button("📥 결과 다운로드", data=img_buf, file_name="team_segmentation_result.png", mime="image/png")

                with st.expander("상세 결과"):
                    for i, (sc, lb) in enumerate(zip(scores, labels)):
                        name = custom_class_names[lb - 1] if 1 <= lb <= len(custom_class_names) else f"cls_{lb}"
                        st.write(f"{i+1}. **{name}** — Score: {sc:.4f}")
    else:
        st.info("먼저 위에서 Fine-tuning을 완료하면 추론이 가능합니다.")

st.markdown("---")
st.caption("Paper: Mask R-CNN (He et al., 2017) | Custom Team Fine-tuning (Optimized) | RoIAlign + AMP")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py 생성 완료 (Cascade 로드 강화 버전)")
print("   - 앱 내부에서 Cascade 강제 다운로드")
print("   - 여러 경로 탐색 + 파일 크기 검증 (>10KB)")
print("   - 로드 성공/실패 명확한 안내")

## Cell 3: Streamlit + Cloudflare Tunnel 실행

In [ ]:
import subprocess
import time
import sys
import os

os.system("pkill -f streamlit || true")
os.system("pkill -f cloudflared || true")
time.sleep(2)

print("🚀 Streamlit 서버 시작 중...")
streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Streamlit 준비 대기 중 (모델 로딩 포함 최대 1~2분)...")
for i in range(50):
    time.sleep(2)
    result = os.system("ss -tuln | grep -q ':8501'")
    if result == 0:
        print("✅ Streamlit ready on port 8501!")
        break
    print(f"   still waiting... ({(i+1)*2}s)")
else:
    print("⚠️ Timeout. Continuing anyway...")

print("\n" + "="*60)
print("🌐 Cloudflare Tunnel 시작...")
print("   아래에 나오는 Public URL을 복사해서 브라우저에서 여세요.")
print("="*60 + "\n")

!cloudflared tunnel --url http://localhost:8501 --no-autoupdate


## (선택) Cleanup - 더 이상 사용하지 않을 때 실행

In [ ]:
!pkill -f streamlit || true
!pkill -f cloudflared || true
print("Processes terminated.")
